In [1]:
import pandas as pd
import cv2
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.blank_detect import is_blank

In [2]:
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "data" / "output" / "dataset" / "labels.csv"
IMG_DIR = BASE_DIR / "data" / "output" / "dataset" / "images"

In [3]:
df = pd.read_csv(CSV_PATH)

images = []
labels = df["is_blank"].values

In [4]:
for filename in df["filename"]:
    img_path = str(IMG_DIR / filename)
    
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue
        
    images.append(img)

In [5]:
X_data = np.array(images)
y_data = labels

print(f"Images array shape: {X_data.shape}")
print(f"Labels array shape: {y_data.shape}")

Images array shape: (500, 500, 500)
Labels array shape: (500,)


In [21]:
df['detected_blank'] = [
    int(is_blank(cv2.imread(str(IMG_DIR / f)), lower_threshold=0.005)) 
    for f in df['filename']
]

df['modified_y'] = (~df['content_type'].isin(['mixed', 'text_only'])).astype(int)

In [22]:
df.head(20)

,filename,is_blank,content_type,detected_blank,modified_y
0,img_0000.png,1,blank,1,1
1,img_0001.png,0,noise_only,1,1
2,img_0002.png,1,blank,1,1
3,img_0003.png,1,blank,1,1
4,img_0004.png,0,noise_only,1,1
5,img_0005.png,1,blank,1,1
6,img_0006.png,0,noise_only,1,1
7,img_0007.png,1,blank,1,1
8,img_0008.png,0,dots_only,1,1
9,img_0009.png,0,noise_only,1,1


In [23]:
y_true = df['modified_y']
y_pred = df['detected_blank']

print("--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Not Blank (0)', 'Blank (1)']))

--- Classification Report ---
               precision    recall  f1-score   support

Not Blank (0)       0.71      0.86      0.78       194
    Blank (1)       0.90      0.78      0.83       306

     accuracy                           0.81       500
    macro avg       0.80      0.82      0.81       500
 weighted avg       0.83      0.81      0.81       500



In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib # For saving the model


from sklearn.model_selection import train_test_split

# Split: 80% Training, 20% Testing
# stratify=y_data keeps the class balance consistent
# random_state=42 ensures you get the same split every time you run it
X_train, X_test, y_train, y_test = train_test_split(
    X_data, 
    y_data, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_data
)

print(f"Train set: {X_train.shape}, {y_train.shape}")
print(f"Test set:  {X_test.shape}, {y_test.shape}")

# Verify the balance (should be ~21% in both)
print(f"Training blank ratio: {y_train.mean():.2%}")
print(f"Testing blank ratio:  {y_test.mean():.2%}")

import numpy as np
import cv2
from sklearn.ensemble import RandomForestClassifier

def extract_features_from_array(img_array):
    ink = np.sum(img_array < 127) / img_array.size
    std = np.std(img_array)
    # edges = cv2.Canny(img_array, 100, 200)
    # edge_dens = np.sum(edges > 0) / edges.size
    
    return [ink, std]

print("Extracting features from training set...")
X_train_features = np.array([extract_features_from_array(img) for img in X_train])

print("Extracting features from test set...")
X_test_features = np.array([extract_features_from_array(img) for img in X_test])

print(f"New X_train shape: {X_train_features.shape}")
print(f"New X_test shape:  {X_test_features.shape}")

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train_features, y_train)

print("Training complete!")

Train set: (400, 500, 500), (400,)
Test set:  (100, 500, 500), (100,)
Training blank ratio: 21.00%
Testing blank ratio:  21.00%
Extracting features from training set...
Extracting features from test set...
New X_train shape: (400, 3)
New X_test shape:  (100, 3)
Training complete!


In [ ]:
from sklearn.metrics import classification_report


y_pred = clf.predict(X_test_features)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        79
           1       1.00      1.00      1.00        21

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [ ]:
# Use the new paths you just defined
test_image_dir = Path("test/dataset/images") # Ensure this matches your generator output
test_csv_path = Path("test/dataset/labels.csv")

# 1. READ THE CORRECT CSV
df_test = pd.read_csv(test_csv_path)

test_images = []
# Use df_test instead of the old df
labels = df_test["is_blank"].values

for filename in df_test["filename"]:
    # 2. USE THE CORRECT IMAGE DIRECTORY
    img_path = str(test_image_dir / filename)
    
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
    
    if img is None:
        print(f"Warning: Could not read {img_path}")
        continue
        
    test_images.append(img)

X_test_data = np.array(test_images)
y_test_data = labels

X_test_features = np.array([extract_features_from_array(img) for img in X_test_data])

print(f"Images array shape: {X_test_data.shape}")
print(f"Labels array shape: {y_test_data.shape}")

Images array shape: (5000, 500, 500)
Labels array shape: (5000,)


In [33]:
y_pred = clf.predict(X_test_features)
print(classification_report(y_test_data, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3990
           1       0.99      1.00      1.00      1010

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

